In [0]:
# =============================================================
# SMOKE TEST 1 — THREE BORROWERS (invoke)
#
# Covers the STANDARD and ENHANCED routes.
# Asserts route, evidence presence, page citations, and output
# contract fields on all three representative borrowers.
# =============================================================

TEST_CASES = [
    {"borrower_id": "B000187", "expected_route": "STANDARD_ANALYST_REVIEW"},
    {"borrower_id": "B001218", "expected_route": "ENHANCED_ANALYST_REVIEW"},
    {"borrower_id": "B005638", "expected_route": "ENHANCED_ANALYST_REVIEW"},
]

TEST_QUESTION = (
    "Assess profitability, leverage and liquidity. "
    "Find evidence for net profit, EBIT, liabilities, "
    "working capital, current assets and equity."
)

for test_case in TEST_CASES:
    graph_output = credit_risk_graph.invoke({
        "borrower_id": test_case["borrower_id"],
        "question":    TEST_QUESTION,
    })

    result       = graph_output["final_result"]
    actual_route = result["policy_assessment"]["workflow_route"]

    assert actual_route == test_case["expected_route"]
    assert result["human_review_required"]      is True
    assert result["automatic_lending_decision"] is False
    assert len(result["document_evidence"]) > 0
    assert all(
        e["page_number"] is not None
        for e in result["document_evidence"]
    )

    print("=" * 75)
    print(f"Borrower    : {result['borrower_id']}")
    print(f"Probability : {result['model_assessment']['probability_of_bankruptcy']:.4f}")
    print(f"Route       : {actual_route}")
    print(f"Rules       : {result['policy_assessment']['triggered_rule_count']}")
    print(f"Evidence    : {len(result['document_evidence'])} chunks")
    for rule in result["policy_assessment"]["triggered_rules"]:
        print(f"  - {rule['rule_id']} [{rule['severity']}]: {rule['message']}")

print("\nSmoke test 1 passed: STANDARD and ENHANCED routes.")

In [0]:
# =============================================================
# SMOKE TEST 2 — DOCUMENT-COMPLETION PATH (stream)
#
# Verifies the architectural gate: when required documents are
# missing, the graph must route to DOCUMENT_COMPLETION_REQUIRED
# and must NOT execute retrieve_evidence.
#
# stream_mode="updates" exposes the exact node execution path.
# No source files or Delta tables are changed.
# =============================================================

original_manifest = document_manifest_df.copy()

try:
    # Simulate a missing financial_summary for B000187.
    document_manifest_df = original_manifest[
        ~(
            (original_manifest["borrower_id"]   == "B000187")
            & (original_manifest["document_type"] == "financial_summary")
        )
    ].copy()

    executed_nodes: list[str] = []
    blocking_result = None

    for update in credit_risk_graph.stream(
        {"borrower_id": "B000187", "question": TEST_QUESTION},
        stream_mode="updates",
    ):
        assert isinstance(update, dict)
        executed_nodes.extend(update.keys())
        if "finalize_case" in update:
            blocking_result = update["finalize_case"]["final_result"]

    assert blocking_result is not None, "Graph did not produce a final result."

    route             = blocking_result["policy_assessment"]["workflow_route"]
    missing_documents = blocking_result["policy_assessment"]["missing_document_types"]
    evidence          = blocking_result["document_evidence"]

    # Validate blocking route
    assert route == "DOCUMENT_COMPLETION_REQUIRED"
    assert "financial_summary" in missing_documents

    # Verify exact execution path
    assert "validate_request"    in executed_nodes
    assert "apply_policy"        in executed_nodes
    assert "document_completion" in executed_nodes
    assert "finalize_case"       in executed_nodes

    # Critical architectural assertion
    assert "retrieve_evidence" not in executed_nodes, (
        "Evidence retrieval should be skipped when required documents are missing."
    )
    assert "enhanced_review" not in executed_nodes
    assert "standard_review" not in executed_nodes

    # Finalize contract on this path
    assert evidence == []
    assert blocking_result["human_review_required"]      is True
    assert blocking_result["automatic_lending_decision"] is False

    print("=" * 75)
    print("Smoke test 2 passed: DOCUMENT_COMPLETION_REQUIRED path")
    print(f"Route          : {route}")
    print(f"Missing docs   : {missing_documents}")
    print(f"Evidence       : {len(evidence)} chunks")
    print(f"Executed nodes : {executed_nodes}")

finally:
    document_manifest_df = original_manifest

In [0]:
# =============================================================
# SMOKE TEST 3 — FOUNDRY CITED SUMMARY (invoke)
#
# Uses B005638 (ENHANCED_ANALYST_REVIEW, 5 triggered rules)
# because it produces the strongest deterministic signal.
#
# Requires: credit_risk_graph (50_langgraph_workflow)
#           generate_foundry_summary() (55_foundry_summary_tool)
# =============================================================

SUMMARY_TEST_INPUT = {
    "borrower_id": "B005638",

    "question": (
        "Assess this borrower's profitability, leverage "
        "and liquidity, and identify the main points that "
        "require human analyst review."
    ),
}

deterministic_graph_output = (
    credit_risk_graph.invoke(
        SUMMARY_TEST_INPUT
    )
)

deterministic_result = (
    deterministic_graph_output[
        "final_result"
    ]
)

summary_result = generate_foundry_summary(
    borrower_id=(
        deterministic_result[
            "borrower_id"
        ]
    ),

    question=(
        deterministic_result[
            "question"
        ]
    ),

    model_assessment=(
        deterministic_result[
            "model_assessment"
        ]
    ),

    policy_assessment=(
        deterministic_result[
            "policy_assessment"
        ]
    ),

    evidence=(
        deterministic_result[
            "document_evidence"
        ]
    ),

    raise_on_error=True,
)

assert (
    summary_result["narrative_status"]
    == "AVAILABLE"
)

assert (
    summary_result["narrative"]
    is not None
)

assert (
    summary_result[
        "generation_metadata"
    ]["response_id"]
)

assert len(summary_result["citations"]) > 0

print("Smoke test 3 passed: Foundry cited-summary")
print(
    "Status      :",
    summary_result["narrative_status"],
)
print(
    "Response ID :",
    summary_result["generation_metadata"]["response_id"],
)
print(
    "Citations   :",
    len(summary_result["citations"]),
)

In [0]:
narrative = summary_result["narrative"]

print("\nEXECUTIVE SUMMARY")
print("-" * 70)
print(narrative["executive_summary"])

print("\nKEY RISK FACTORS")
print("-" * 70)

for finding in narrative["key_risk_factors"]:
    print(
        f"- {finding['statement']} "
        f"{finding['evidence_ids']}"
    )

print("\nMITIGATING FACTORS")
print("-" * 70)

if narrative["mitigating_factors"]:
    for finding in narrative["mitigating_factors"]:
        print(
            f"- {finding['statement']} "
            f"{finding['evidence_ids']}"
        )
else:
    print("- None supported by the retrieved evidence.")

print("\nANALYST REVIEW ACTIONS")
print("-" * 70)

for action in narrative["analyst_review_actions"]:
    print(f"- {action}")

print("\nLIMITATIONS")
print("-" * 70)

for limitation in narrative["limitations"]:
    print(f"- {limitation}")

print("\nTRUSTED CITATIONS")
print("-" * 70)

for citation in summary_result["citations"]:
    print(
        f"- {citation['evidence_id']}: "
        f"{citation['document_id']}, "
        f"page {citation['page_number']}, "
        f"section={citation['section']}"
    )

In [0]:
# =============================================================
# SMOKE TEST 4 — SAFE FOUNDRY FAILURE
#
# Verifies:
# - A Foundry/API failure does not raise into the workflow
# - A controlled UNAVAILABLE result is returned
# - No narrative or citations are fabricated
# - Deterministic inputs remain unchanged
# - The valid deployment is always restored
# =============================================================

from copy import deepcopy


# Use the successful deterministic_result already created
# in smoke test 3.
model_before = deepcopy(
    deterministic_result["model_assessment"]
)

policy_before = deepcopy(
    deterministic_result["policy_assessment"]
)

evidence_before = deepcopy(
    deterministic_result["document_evidence"]
)


original_deployment = FOUNDRY_DEPLOYMENT

try:
    # Deliberately use a non-existent deployment to trigger
    # a controlled Foundry API failure.
    FOUNDRY_DEPLOYMENT = (
        "temporary-invalid-deployment-for-fallback-test"
    )

    fallback_result = generate_foundry_summary(
        borrower_id=(
            deterministic_result["borrower_id"]
        ),

        question=(
            deterministic_result["question"]
        ),

        model_assessment=(
            deterministic_result["model_assessment"]
        ),

        policy_assessment=(
            deterministic_result["policy_assessment"]
        ),

        evidence=(
            deterministic_result["document_evidence"]
        ),

        # Critical: production graph behaviour
        raise_on_error=False,
    )

    # ---------------------------------------------------------
    # Validate safe fallback contract
    # ---------------------------------------------------------
    assert (
        fallback_result["narrative_status"]
        == "UNAVAILABLE"
    )

    assert fallback_result["narrative"] is None

    assert fallback_result["citations"] == []

    assert fallback_result["fallback_message"]

    metadata = fallback_result[
        "generation_metadata"
    ]

    assert metadata["response_id"] is None
    assert metadata["error_type"] is not None
    assert metadata["provider"] == "Microsoft Foundry"

    # ---------------------------------------------------------
    # Verify deterministic inputs were not modified
    # ---------------------------------------------------------
    assert (
        deterministic_result["model_assessment"]
        == model_before
    )

    assert (
        deterministic_result["policy_assessment"]
        == policy_before
    )

    assert (
        deterministic_result["document_evidence"]
        == evidence_before
    )

    print("Smoke test 4 passed: Foundry safe-failure")
    print(
        "Narrative status :",
        fallback_result["narrative_status"],
    )
    print(
        "Error type       :",
        metadata["error_type"],
    )
    print(
        "Narrative        :",
        fallback_result["narrative"],
    )
    print(
        "Citations        :",
        len(fallback_result["citations"]),
    )
    print(
        "Fallback message :",
        fallback_result["fallback_message"],
    )

finally:
    # Always restore the real deployment, even if an
    # assertion unexpectedly fails.
    FOUNDRY_DEPLOYMENT = original_deployment


assert FOUNDRY_DEPLOYMENT == original_deployment

print(
    "Original Foundry deployment restored:",
    FOUNDRY_DEPLOYMENT,
)

In [0]:
# =============================================================
# SMOKE TEST 5 — FOUNDRY DOCUMENT-COMPLETION GUARD
#
# Verifies:
# - Foundry is skipped for incomplete cases
# - No narrative or citations are generated
# - No API request is made
# =============================================================

document_completion_result = generate_foundry_summary(
    borrower_id="B000187",

    question=(
        "Assess profitability, leverage and liquidity."
    ),

    model_assessment={
        "probability_of_bankruptcy": 0.043381,
        "review_threshold": 0.068990,
        "review_required": False,
        "decision_support_status": "BELOW_REVIEW_THRESHOLD",
    },

    policy_assessment={
        "workflow_route": "DOCUMENT_COMPLETION_REQUIRED",
        "route_reason": (
            "A required financial document is missing."
        ),
        "workflow_message": (
            "Document completion is required before assessment."
        ),
        "triggered_rule_count": 1,
        "triggered_rules": [
            {
                "rule_id": "DOC-001",
                "severity": "BLOCKING",
                "category": "DOCUMENT_COMPLETENESS",
                "message": (
                    "Required documents are missing: "
                    "financial_summary"
                ),
            }
        ],
    },

    evidence=[],

    raise_on_error=True,
)

assert (
    document_completion_result["narrative_status"]
    == "SKIPPED_DOCUMENT_COMPLETION_REQUIRED"
)

assert document_completion_result["narrative"] is None
assert document_completion_result["citations"] == []
assert (
    document_completion_result[
        "generation_metadata"
    ]["response_id"]
    is None
)

print("Smoke test 5 passed: document-completion guard")
print(
    "Narrative status :",
    document_completion_result["narrative_status"],
)
print(
    "Narrative        :",
    document_completion_result["narrative"],
)
print(
    "Citations        :",
    len(document_completion_result["citations"]),
)

In [0]:
# =============================================================
# APPLICATION INSIGHTS TELEMETRY SMOKE TEST
#
# Sends no borrower data, question, evidence or model output.
# =============================================================

from opentelemetry.trace import (
    SpanKind,
    Status,
    StatusCode,
)


with credit_risk_tracer.start_as_current_span(
    "credit_risk_copilot.telemetry_smoke_test",
    kind=SpanKind.INTERNAL,
) as span:

    span.set_attribute(
        "app.component",
        "tracing_setup",
    )

    span.set_attribute(
        "app.test_type",
        "telemetry_smoke_test",
    )

    span.set_attribute(
        "data.synthetic",
        True,
    )

    span.set_attribute(
        "content.recorded",
        False,
    )

    span.set_status(
        Status(
            StatusCode.OK
        )
    )


flush_result = flush_credit_risk_telemetry()

print("Telemetry smoke test span created.")
print("Flush requested:", flush_result)

In [0]:
# =============================================================
# END-TO-END FOUNDRY LANGGRAPH TEST
#
# Tests the complete integrated path:
#   validate_request → apply_policy → retrieve_evidence
#   → enhanced_review → generate_cited_summary
#   → finalize_case
#
# Makes a real paid Foundry API request.
# Run manually — NOT part of automatic bootstrap.
#
# Guard: set env var CREDIT_RISK_RUN_E2E=1 to enable.
# When run via %run ./60_smoke_tests (bootstrap), this test
# is automatically skipped.
# =============================================================

import os as _os


def _run_e2e_foundry_langgraph_test() -> None:

    END_TO_END_INPUT = {
        "borrower_id": "B005638",
        "question": (
            "Assess this borrower's profitability, leverage "
            "and liquidity, and identify the main points that "
            "require human analyst review."
        ),
    }

    graph_output = credit_risk_graph.invoke(
        END_TO_END_INPUT
    )

    result = graph_output[
        "final_result"
    ]

    narrative_result = result[
        "narrative_assessment"
    ]

    assert (
        result[
            "policy_assessment"
        ]["workflow_route"]
        == "ENHANCED_ANALYST_REVIEW"
    )

    assert (
        narrative_result[
            "narrative_status"
        ]
        == "AVAILABLE"
    )

    assert (
        narrative_result[
            "narrative"
        ]
        is not None
    )

    assert len(
        narrative_result[
            "citations"
        ]
    ) > 0

    assert result[
        "human_review_required"
    ] is True

    assert result[
        "automatic_lending_decision"
    ] is False

    print(
        "End-to-end Foundry LangGraph test: passed"
    )
    print(
        "Borrower         :",
        result["borrower_id"],
    )
    print(
        "Route            :",
        result[
            "policy_assessment"
        ]["workflow_route"],
    )
    print(
        "Narrative status :",
        narrative_result[
            "narrative_status"
        ],
    )
    print(
        "Citations        :",
        len(
            narrative_result[
                "citations"
            ]
        ),
    )
    print(
        "Response ID      :",
        narrative_result[
            "generation_metadata"
        ]["response_id"],
    )


if _os.getenv("CREDIT_RISK_RUN_E2E", "0") == "1":
    _run_e2e_foundry_langgraph_test()
else:
    print("End-to-end Foundry LangGraph test: SKIPPED")
    print(
        "(set env var CREDIT_RISK_RUN_E2E=1 "
        "to run the paid Foundry test)"
    )